# Projeto Final — Pipeline ETL com os dados de Viagens a Serviço

**Análise de Dados com Python — Turma T5 (SENAI / LAB365) · Módulo 1, Semana 13**

Esta pasta é o **projeto pronto**: um pipeline de dados em **Python + PostgreSQL** que baixa os dados abertos de *Viagens a Serviço* do Portal da Transparência, guarda o dado bruto (camada **Raw**), limpa e tipa (camada **Silver**) e responde perguntas de negócio com tabelas e gráficos (camada **Gold**).

Este notebook é o **roteiro para executar o projeto no seu computador**, clicando célula por célula. Você não precisa escrever código: só trocar **uma linha** (a sua senha do PostgreSQL) no Passo 2.

| Passo | O que acontece | Arquivo do projeto que roda |
|---|---|---|
| 1 | instala as bibliotecas | `requirements.txt` |
| 2 | grava a sua senha no arquivo `.env` e testa a conexão | `.env.example` → `.env` |
| 3 | cria o banco `transparencia` e as 8 tabelas | `0_criar_banco.sql` |
| 4 | baixa os dados e carrega a camada Raw | `1_extrair.py` |
| 5 | converte os tipos e carrega a camada Silver | `2_transformar.py` |
| 6 | cria a camada Gold e responde as 7 perguntas | `3_analise.ipynb` |
| 7 | confere todas as tabelas | — |

### Antes de começar

- **Onde executar:** no **VS Code**, no seu computador. Este projeto **não roda no Google Colab**, porque precisa do PostgreSQL instalado na sua máquina (o mesmo das Semanas 08 a 12).
- **Abra a pasta certa:** menu **File → Open Folder** e escolha a pasta `T5_Projeto_Final/Aluno`. Os scripts procuram os outros arquivos nesta mesma pasta.
- **Escolha o kernel:** clique em **Select Kernel** (canto superior direito) → **Python Environments** → o Python instalado na sua máquina.
- **Tempo total:** cerca de **5 minutos** (o download tem uns 160 MB e as cargas têm quase 1,9 milhão de linhas). Enquanto uma célula roda, aparece um relógio ao lado dela: espere terminar antes de executar a próxima.
- **Pode executar de novo quando quiser:** todas as etapas apagam o conteúdo antigo antes de carregar, então nada fica duplicado.

## Passo 1 — Instalando as bibliotecas

### 📖 Antes do código

O arquivo `requirements.txt` lista as bibliotecas do projeto: `psycopg2-binary` (conexão com o PostgreSQL), `pandas` (tabelas), `matplotlib` (gráficos), `requests` (download pela internet) e `ipykernel` (o que faz o notebook rodar no VS Code). O comando `%pip install -r requirements.txt` instala **todas as bibliotecas da lista** de uma vez, no Python deste notebook.

In [ ]:
%pip install -r requirements.txt

**Deu certo?** As últimas linhas mostram `Successfully installed ...` ou `Requirement already satisfied ...`. Se instalou algo agora, clique em **Restart** na barra do notebook antes de continuar.

## Passo 2 — A sua senha do PostgreSQL no arquivo `.env`

### 📖 Antes do código

A senha **nunca** fica escrita no código dos scripts: ela fica no arquivo `.env`, que o `.gitignore` impede de ir para o GitHub. O projeto traz o modelo `.env.example`, com o texto `sua_senha_aqui` no lugar da senha.

A célula abaixo lê o modelo com `open(...).read()`, troca `sua_senha_aqui` pela sua senha com o método de texto `.replace()` e grava o resultado num arquivo novo, o `.env`, com `open(".env", "w")` (o `"w"` abre o arquivo para escrita e cria o arquivo se ele não existir).

**✏️ A única coisa que você edita no projeto:** na primeira linha, troque `SUA_SENHA_AQUI` pela senha que você definiu ao instalar o PostgreSQL, **mantendo as aspas**. Exemplo: `SENHA = "123456"`.

In [ ]:
SENHA = "SUA_SENHA_AQUI"

modelo = open(".env.example", encoding="utf-8").read()
conteudo = modelo.replace("sua_senha_aqui", SENHA)

with open(".env", "w", encoding="utf-8") as arquivo:
    arquivo.write(conteudo)

print("Arquivo .env gravado nesta pasta.")

### 📖 Antes do código — testando a conexão

Antes de rodar o pipeline, a célula abaixo testa a senha. O `psycopg2.connect(...)` tenta abrir uma conexão com o PostgreSQL no banco `postgres`, que existe em toda instalação. Se abrir, aparece `Conexão OK`; se não abrir, o `except` mostra o erro, sem travar o notebook.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(host="localhost", port=5432, dbname="postgres", user="postgres", password=SENHA)
    print("Conexão OK. O PostgreSQL aceitou a sua senha.")
    conexao.close()
except Exception as erro:
    print("Não foi possível conectar.")
    print("Tipo do erro:", type(erro).__name__)
    print("Mensagem:", erro)

**Apareceu `Conexão OK`?** Siga para o Passo 3.

**Apareceu um erro?** Os dois casos mais comuns:

| O que apareceu | Causa | O que fazer |
|---|---|---|
| `UnicodeDecodeError` (com `byte 0xe7`) | **senha errada**: o PostgreSQL em português respondeu "falha na autenticação", e o `0xe7` é o **ç** de "autenticação" | corrija a senha no Passo 2, execute as duas células de novo e clique em **Restart** antes de continuar |
| `OperationalError` com `Connection refused` | o PostgreSQL **não está ligado** | no Windows, abra **Serviços**, procure `postgresql` e clique em **Iniciar** |

## Passo 3 — Fase 0: o banco `transparencia` e as 8 tabelas (`0_criar_banco.sql`)

### 📖 Antes do código

O arquivo `0_criar_banco.sql` tem **duas partes**, porque o comando `CREATE DATABASE` cria o banco mas não entra nele:

- **Parte 1** (conectado no banco `postgres`): cria o banco `transparencia`. O `CREATE DATABASE` não pode rodar dentro de uma transação, por isso a conexão usa `conexao.autocommit = True` (cada comando é confirmado na hora). Antes de criar, a consulta `SELECT 1 FROM pg_database WHERE datname = ...` verifica se o banco já existe, para a célula poder ser executada de novo sem erro.
- **Parte 2** (conectado no banco `transparencia`): cria as 4 tabelas **Raw** (tudo texto, cópia fiel do CSV) e as 4 tabelas **Silver** (tipadas, com `PRIMARY KEY`, `FOREIGN KEY`, `NOT NULL`, `CHECK` e `UNIQUE`).

A célula lê o arquivo `.sql` como texto e usa o método `.split("PARTE 2")` para cortar o texto no marcador `PARTE 2`: o `[-1]` fica com o último pedaço, que é o SQL da Parte 2. O `from config import POSTGRES_CONFIG` traz os dados de conexão lidos do `.env`, e o `conectar()` do arquivo `banco.py` abre a conexão com o banco do projeto.

In [ ]:
import psycopg2

from banco import conectar
from config import POSTGRES_CONFIG

NOME_BANCO = POSTGRES_CONFIG["dbname"]

# Parte 1: cria o banco (conectado no banco "postgres")
conexao = psycopg2.connect(host=POSTGRES_CONFIG["host"], port=POSTGRES_CONFIG["port"], dbname="postgres",
                           user=POSTGRES_CONFIG["user"], password=POSTGRES_CONFIG["password"])
conexao.autocommit = True
cursor = conexao.cursor()
cursor.execute("SELECT 1 FROM pg_database WHERE datname = %s", (NOME_BANCO,))
if cursor.fetchone() is None:
    cursor.execute(f"CREATE DATABASE {NOME_BANCO}")
    print(f"Banco '{NOME_BANCO}' criado.")
else:
    print(f"O banco '{NOME_BANCO}' já existia.")
conexao.close()

# Parte 2: executa o restante do 0_criar_banco.sql (conectado no banco do projeto)
texto_sql = open("0_criar_banco.sql", encoding="utf-8").read()
parte_2 = texto_sql.split("PARTE 2")[-1]
parte_2 = parte_2.split("\n", 1)[1]          # descarta o resto da linha do marcador

conexao = conectar()
cursor = conexao.cursor()
cursor.execute(parte_2)
conexao.commit()

cursor.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name
""")
tabelas = [linha[0] for linha in cursor.fetchall()]
conexao.close()
print(f"{len(tabelas)} tabelas no banco:", tabelas)

**Resultado esperado:** `8 tabelas no banco`, 4 com o prefixo `raw_` e 4 com o prefixo `silver_`. Ainda estão **vazias**: os próximos passos carregam os dados. (Se você já tinha executado o `3_analise.ipynb` antes, o `0_criar_banco.sql` apaga também as tabelas e *views* Gold, que dependem das Silver.)

> **Outra forma de fazer o mesmo passo:** abrir o `0_criar_banco.sql` na extensão PostgreSQL do VS Code e executar a Parte 1 conectado no banco `postgres` e a Parte 2 conectado no banco `transparencia`. O resultado é idêntico.

## Passo 4 — Fase 1: download e camada Raw (`1_extrair.py`)

### 📖 Antes do código

O comando `%run 1_extrair.py` executa o script **inteiro**, aqui mesmo no notebook, e mostra as mensagens que ele imprime. O que o script faz:

1. **Baixa** o arquivo `.zip` do Google Drive com a biblioteca `requests` e descompacta os 4 CSVs na pasta `data/` com o módulo `zipfile`. Se os CSVs já estiverem em `data/`, ele não baixa de novo.
2. **Lê cada CSV em blocos** de 50 mil linhas (`pd.read_csv(..., chunksize=50_000)`), para não carregar centenas de megabytes na memória de uma vez. Tudo é lido como texto (`dtype=str`), sem nenhuma conversão.
3. **Esvazia** cada tabela Raw com `TRUNCATE` e **grava** os blocos com `execute_values`, que manda milhares de linhas num único `INSERT`.
4. **Confere** as linhas lidas do CSV com as linhas gravadas na tabela.

Se algo falhar, o `try/except` mostra o erro e o `rollback()` desfaz a carga incompleta.

In [ ]:
%run 1_extrair.py

**Resultado esperado:** no fim, a conferência mostra `OK` nas 4 tabelas: **341.860** viagens, **606.916** pagamentos, **167.260** passagens e **763.349** trechos. A pasta `data/` agora tem os 4 CSVs (e ela **nunca** vai para o GitHub: está no `.gitignore`).

## Passo 5 — Fase 2: da Raw para a Silver (`2_transformar.py`)

### 📖 Antes do código

O comando `%run 2_transformar.py` executa a transformação. O script tem **funções de conversão** (o enunciado pede funções específicas):

| Função | Converte | Exemplo |
|---|---|---|
| `texto_para_decimal()` | valores e taxas | `"1272,97"` → `1272.97` (troca a vírgula por ponto e usa `pd.to_numeric`) |
| `texto_para_data()` | datas | `"17/09/2024"` → data (usa `pd.to_datetime` com `format="%d/%m/%Y"`) |
| `limpar_texto()` | textos | tira espaços das pontas; texto vazio vira `NULL` |

Na `silver_viagem`, o script também **calcula** duas colunas: `valor_total` = diárias + passagens + outros gastos − devolução, e `duracao_dias` = (data de fim − data de início) + 1.

A `silver_viagem` é carregada **primeiro**, porque as outras 3 tabelas têm `FOREIGN KEY` apontando para ela. No fim, o script confere a Raw com a Silver: mesma quantidade de linhas e mesma soma dos pagamentos.

In [ ]:
%run 2_transformar.py

**Resultado esperado:** a conferência mostra `OK` em todas as linhas: as mesmas quantidades da Raw, as **664** datas de emissão vazias viraram `NULL`, e a soma dos pagamentos é **idêntica** nas duas camadas (R$ 1.194.365.457,37).

## Passo 6 — Fase 3: camada Gold e as 7 perguntas (`3_analise.ipynb`)

A análise fica num notebook próprio, porque ela é **para ser lida**: cada pergunta tem a consulta SQL, a tabela, o gráfico e a conclusão.

1. No painel **Explorer** do VS Code (à esquerda), abra o arquivo **`3_analise.ipynb`**.
2. Escolha o mesmo kernel deste notebook (**Select Kernel**).
3. Clique em **Run All**.
4. Leia cada pergunta e a conclusão abaixo do gráfico.

O `3_analise.ipynb` também cria a camada Gold: as tabelas `gold_pagamento_resumo` e `gold_trecho_resumo` (com `JOIN` + `GROUP BY`) e as *views* `vw_gold_pagamento_resumo` e `vw_gold_trecho_resumo`.

**Depois de executar o `3_analise.ipynb`, volte aqui e execute o Passo 7.**

## Passo 7 — Conferência final: tudo no lugar?

### 📖 Antes do código

A célula conta as linhas de cada tabela do projeto com `SELECT COUNT(*)`. Antes de contar, a função `to_regclass()` do PostgreSQL confere se a tabela existe: ela devolve o nome da tabela, ou `NULL` (que chega ao Python como `None`) quando a tabela não existe. Assim, se você ainda não executou o `3_analise.ipynb`, a célula avisa em vez de dar erro.

In [ ]:
from banco import conectar

TABELAS_DO_PROJETO = [
    "raw_viagem", "raw_pagamento", "raw_passagem", "raw_trecho",
    "silver_viagem", "silver_pagamento", "silver_passagem", "silver_trecho",
    "gold_pagamento_resumo", "gold_trecho_resumo",
    "vw_gold_pagamento_resumo", "vw_gold_trecho_resumo",
]

conexao = conectar()
cursor = conexao.cursor()
for tabela in TABELAS_DO_PROJETO:
    cursor.execute("SELECT to_regclass(%s)", (tabela,))
    if cursor.fetchone()[0] is None:
        print(f"{tabela:<26} ainda não existe (execute o 3_analise.ipynb)")
    else:
        cursor.execute(f"SELECT COUNT(*) FROM {tabela}")
        print(f"{tabela:<26} {cursor.fetchone()[0]:>9} linhas")
conexao.close()

**Resultado esperado:** as 4 tabelas Raw e as 4 Silver com as mesmas quantidades (341.860 · 606.916 · 167.260 · 763.349), e as 2 tabelas Gold com a mesma quantidade de linhas das suas *views*.

### ✅ Projeto executado

Você executou o pipeline completo: **Fase 0** (banco e tabelas), **Fase 1** (download + Raw), **Fase 2** (Silver tipada) e **Fase 3** (Gold, perguntas e gráficos). O `README.md` desta pasta explica as decisões do projeto, mostra as respostas das 7 perguntas e lista as melhorias possíveis.

---
*Prof. Especialista Cláudio F. Neves*